# 🤖 Evaluasi Kinerja Auto-Scaler
Meniru evaluasi elastisitas Auto-Scaler. Kami memuat hasil prediksi yang dihasilkan masing-masing model untuk disimulasikan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Load Real Test Data & Scaler
data = np.load('processed_data.npz')
y_test = data['y_test'] # shape (N, 2)

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)

# Load Predictions if available
model_names = ['LSTM', 'GRU', 'BiLSTM', 'DUCFF', 'True_DUCFF', 'ARIMA']
predictions = {}
for name in model_names:
    if os.path.exists(f'pred_{name}.npy'):
        # reshape back to (N, 2) because it was saved without flattening or saved with flattening
        pred_flat = np.load(f'pred_{name}.npy')
        if len(pred_flat.shape) == 1:
            predictions[name] = pred_flat.reshape(-1, 2)
        else:
            predictions[name] = pred_flat
        print(f"Loaded predictions for {name}")
    else:
        print(f"File pred_{name}.npy not found. Skip.")



In [ ]:
y_test_real = rps_scaler.inverse_transform(y_test) # (N, 2)
y_test_media = y_test_real[:, 0]
y_test_content = y_test_real[:, 1]

RPS_CAPACITY_PER_CONTAINER = 50.0
R_min = 5
CDT_MAX = 10  
SDR = 0.40  

def simulate_autoscaler(predicted_rps, actual_rps):
    R_current = R_min; CDT = 0; supply_history = []; demand_history = []
    for t in range(len(predicted_rps)):
        R_demand = max(int(np.ceil(actual_rps[t] / RPS_CAPACITY_PER_CONTAINER)), R_min)
        demand_history.append(R_demand)
        R_estimated = max(int(np.ceil(predicted_rps[t] / RPS_CAPACITY_PER_CONTAINER)), R_min)
        
        if R_estimated > R_current:
            R_current = R_estimated
            CDT = CDT_MAX
        elif R_estimated < R_current:
            if CDT <= 0:
                CDT = CDT_MAX
                R_current = max(R_current - int(np.floor((R_current - R_estimated) * (1 - SDR))), R_estimated, R_min)
            else:
                CDT -= 1
        else:
            if CDT > 0: CDT -= 1
        supply_history.append(R_current)
    return np.array(supply_history), np.array(demand_history)

total_T = len(y_test_real)

# Reactive metrics for Media
demand_real_media = np.array([max(int(np.ceil(d / RPS_CAPACITY_PER_CONTAINER)), R_min) for d in y_test_media])
reactive_supply_media = np.zeros(total_T)
reactive_supply_media[0] = R_min
for t in range(1, total_T):
    reactive_supply_media[t] = max(int(np.ceil(y_test_media[t-1] / RPS_CAPACITY_PER_CONTAINER)), R_min)

# Reactive metrics for Content
demand_real_content = np.array([max(int(np.ceil(d / RPS_CAPACITY_PER_CONTAINER)), R_min) for d in y_test_content])
reactive_supply_content = np.zeros(total_T)
reactive_supply_content[0] = R_min
for t in range(1, total_T):
    reactive_supply_content[t] = max(int(np.ceil(y_test_content[t-1] / RPS_CAPACITY_PER_CONTAINER)), R_min)

def calc_metrics(supply, demand):
    under = np.maximum(demand - supply, 0)
    over = np.maximum(supply - demand, 0)
    safe_demand = np.where(demand == 0, 1, demand)
    return ((100.0/total_T)*np.sum(under/safe_demand), (100.0/total_T)*np.sum(over/safe_demand), (np.sum(under>0)/total_T)*100.0, (np.sum(over>0)/total_T)*100.0)

theta_U_n_m, theta_O_n_m, T_U_n_m, T_O_n_m = calc_metrics(reactive_supply_media, demand_real_media)
autoscaler_metrics_media = {'Reactive': {'θ_U': theta_U_n_m, 'θ_O': theta_O_n_m, 'T_U': T_U_n_m, 'T_O': T_O_n_m, 'η (Speedup)': 1.00}}

theta_U_n_c, theta_O_n_c, T_U_n_c, T_O_n_c = calc_metrics(reactive_supply_content, demand_real_content)
autoscaler_metrics_content = {'Reactive': {'θ_U': theta_U_n_c, 'θ_O': theta_O_n_c, 'T_U': T_U_n_c, 'T_O': T_O_n_c, 'η (Speedup)': 1.00}}

all_supplies_media = {}
all_supplies_content = {}

if len(predictions) > 0:
    for name, pred in predictions.items():
        pred_real = rps_scaler.inverse_transform(pred) # (N, 2)
        pred_media = pred_real[:, 0]
        pred_content = pred_real[:, 1]
        
        # Sim Media
        supp_m, dem_m = simulate_autoscaler(pred_media, y_test_media)
        all_supplies_media[name] = supp_m
        t_U_a, t_O_a, TU_a, TO_a = calc_metrics(supp_m, dem_m)
        eta = ((theta_U_n_m/t_U_a if t_U_a>0 else 1)*(theta_O_n_m/t_O_a if t_O_a>0 else 1)*(T_U_n_m/TU_a if TU_a>0 else 1)*(T_O_n_m/TO_a if TO_a>0 else 1))**0.25
        autoscaler_metrics_media[name] = {'θ_U': t_U_a, 'θ_O': t_O_a, 'T_U': TU_a, 'T_O': TO_a, 'η (Speedup)': eta}
        
        # Sim Content
        supp_c, dem_c = simulate_autoscaler(pred_content, y_test_content)
        all_supplies_content[name] = supp_c
        t_U_a_c, t_O_a_c, TU_a_c, TO_a_c = calc_metrics(supp_c, dem_c)
        eta_c = ((theta_U_n_c/t_U_a_c if t_U_a_c>0 else 1)*(theta_O_n_c/t_O_a_c if t_O_a_c>0 else 1)*(T_U_n_c/TU_a_c if TU_a_c>0 else 1)*(T_O_n_c/TO_a_c if TO_a_c>0 else 1))**0.25
        autoscaler_metrics_content[name] = {'θ_U': t_U_a_c, 'θ_O': t_O_a_c, 'T_U': TU_a_c, 'T_O': TO_a_c, 'η (Speedup)': eta_c}

    print("--- Media Service Metrics ---")
    display(pd.DataFrame(autoscaler_metrics_media).T)
    print("\n--- Content Service Metrics ---")
    display(pd.DataFrame(autoscaler_metrics_content).T)
else:
    print("Prediksi belum ada, silahkan jalankan notebook model terlebih dahulu.")



## Total Started Containers (Overhead / API Oscillations)

In [ ]:
if len(all_supplies_media) > 0:
    started_containers_m = {}
    started_containers_c = {}
    for name in all_supplies_media.keys():
        supp_m = all_supplies_media[name]
        supp_c = all_supplies_content[name]
        started_containers_m[name] = sum((supp_m[i] - supp_m[i-1]) for i in range(1, len(supp_m)) if supp_m[i] > supp_m[i-1])
        started_containers_c[name] = sum((supp_c[i] - supp_c[i-1]) for i in range(1, len(supp_c)) if supp_c[i] > supp_c[i-1])

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    sns.barplot(x=list(started_containers_m.keys()), y=list(started_containers_m.values()), palette='coolwarm', ax=axes[0])
    axes[0].set_title('Media: Total Containers Started (Overhead)', fontsize=14)
    
    sns.barplot(x=list(started_containers_c.keys()), y=list(started_containers_c.values()), palette='coolwarm', ax=axes[1])
    axes[1].set_title('Content: Total Containers Started (Overhead)', fontsize=14)
    plt.show()



## Qualitative Analysis of Demand vs Supply (GDS Algorithm)

In [ ]:
if len(all_supplies_media) > 0:
    fig, axes = plt.subplots(len(all_supplies_media), 2, figsize=(16, 5 * len(all_supplies_media)), sharex=True)
    plot_len = 400
    
    if len(all_supplies_media) == 1:
        axes = [axes]

    for i, m_name in enumerate(all_supplies_media.keys()):
        supp_m = all_supplies_media[m_name]
        supp_c = all_supplies_content[m_name]
        
        # Plot Media
        axes[i][0].plot(demand_real_media[:plot_len], label='Demand (Red)', color='red', linestyle='--')
        axes[i][0].plot(supp_m[:plot_len], label=f'Supply {m_name} (Blue)', color='blue', drawstyle='steps-post', alpha=0.7)
        axes[i][0].set_title(f'Media ({m_name}): Supply vs Demand', fontsize=12)
        axes[i][0].legend()
        
        # Plot Content
        axes[i][1].plot(demand_real_content[:plot_len], label='Demand (Red)', color='red', linestyle='--')
        axes[i][1].plot(supp_c[:plot_len], label=f'Supply {m_name} (Blue)', color='green', drawstyle='steps-post', alpha=0.7)
        axes[i][1].set_title(f'Content ({m_name}): Supply vs Demand', fontsize=12)
        axes[i][1].legend()

    plt.tight_layout()
    plt.show()

